# Experiment B — Dissipation-Anomaly Inconsistency Detector

## Goal

Test whether the **energy-dissipation anomaly** $\Delta E_{\text{anomaly}}$ and the
**curvature proxy** $\mathcal{K}_{\max}$ — both native, parameter-free signals of the
conservative architecture — can detect when a continuation is *inconsistent* with its
context, and whether they **beat the softmax-entropy baseline** $H_{\text{softmax}}$.

### Why a controlled "shuffle" task (not TriviaQA)

The SPLM-family checkpoints are trained on **TinyStories** and have no world knowledge,
so factual-QA hallucination labels would just measure "this model never knew facts".
Instead we use a **controlled story-consistency** task:

- **Clean (label 0):** prompt + the gold continuation from the same story.
- **Corrupted (label 1):** prompt + the **shuffled** gold continuation.

Shuffling permutes the continuation tokens, so the **token multiset is identical** to the
clean case — only the *sequential coherence* is destroyed. This is the hardest *fair* test:
the unigram statistics cannot trivially separate the classes, so any detector must use the
trajectory geometry. The design is **paired**: each story yields one clean and one shuffled
example, differing only in continuation order. Evaluation is **teacher-forced** (no generation).

### The signals (per continuation token, then averaged per example)

For each token's depth trajectory $h_0 \to h_L$ through the $L$ layers:

- $\Delta E_{\text{anomaly}} = \frac{1}{L}\sum_\ell \big| \Delta E_{\text{obs}}(\ell) - \Delta E_{\text{expected}}(\ell) \big|$,
  with $\Delta E_{\text{expected}}(\ell) = -\gamma_{\text{eff}} \lVert v_\ell \rVert^2 \, dt$
- $\mathcal{K}_{\max} = \lambda_{\max}(\nabla^2 V_\theta) / (2 T_\ell)$ at the mid-layer (batched HVP power iteration)
- $H_{\text{softmax}} = -\sum_v p_v \log p_v$ (next-token entropy — the attention baseline)

### Critical: $\gamma_{\text{eff}}$, not $\gamma_{\text{param}}$

The expected-dissipation baseline depends on $\gamma$. The learned-$\gamma$ diagnostic
showed LayerNorm acts as a counter-damping force, so the model's **effective** damping
$\gamma_{\text{eff}} \approx 0.13$ is $\sim 6\times$ weaker than the stored parameter
$\gamma_{\text{param}} \approx 0.93$. We measure $\gamma_{\text{eff}}$ per model from the
held-out T-ratio profile (Cell 4) and use it for $\Delta E_{\text{expected}}$.

## Models Under Test

| Model | Checkpoint |
|-------|-----------|
| Multi-Xi SPLM | `dimitarpg13/semsimula-splm-multixi` |
| Fock v2.1 PARFLM | `dimitarpg13/semsimula-fock-parflm` |
| Fock Attention PARFLM | `dimitarpg13/semsimula-fock-attention` |

## Hypothesis

$$\text{AUROC}(\Delta E_{\text{anomaly}}) > \text{AUROC}(H_{\text{softmax}}), \qquad
  \text{AUROC}([\Delta E_{\text{anomaly}}, \mathcal{K}_{\max}]) = \text{strongest}$$

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # TF32 OFF: autograd.grad through V_theta needs full fp32 precision for
    # stable Hessian-vector products (K_max) and grad-V (energy anomaly).
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_hallucination_detector')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_hallucination_detector'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print(f'Drive root  : {DRIVE_ROOT}')
print(f'Results dir : {DRIVE_RESULTS}')

In [ ]:
# ── Cell 2: Download checkpoints & define model registry ──────────
# Models are loaded ONE AT A TIME during each stage to keep peak RAM low.
from huggingface_hub import hf_hub_download

CKPT_DIR = 'checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

HF_CKPTS = {
    'splm':      ('dimitarpg13/semsimula-splm-multixi',   'checkpoint/model_16k.pt'),
    'fock_v21':  ('dimitarpg13/semsimula-fock-parflm',    'checkpoint/model.pt'),
    'fock_attn': ('dimitarpg13/semsimula-fock-attention', 'checkpoint/model.pt'),
}

ckpt_paths = {}
for name, (repo, fname) in HF_CKPTS.items():
    print(f'Downloading {name} from {repo}...')
    path = hf_hub_download(repo_id=repo, filename=fname, local_dir=CKPT_DIR)
    ckpt_paths[name] = path
    print(f'  -> {path}')

# Precompute logfreq surprisal if not cached (used by V_theta mass term)
SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')
if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )
    assert os.path.exists(LOGFREQ_PATH)
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')


def _load_from_ckpt(ckpt_path, ConfigClass, ModelClass, logfreq_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg_dict = ckpt.get('config') or ckpt.get('model_cfg') or ckpt.get('cfg', {})
    if isinstance(cfg_dict, dict):
        known = {f.name for f in dc_fields(ConfigClass)}
        cfg = ConfigClass(**{k: v for k, v in cfg_dict.items() if k in known})
    else:
        cfg = cfg_dict
    if hasattr(cfg, 'logfreq_path'):
        cfg.logfreq_path = logfreq_path
    model = ModelClass(cfg)
    sd = ckpt.get('model_state_dict') or ckpt.get('state_dict') or ckpt
    model.load_state_dict(sd, strict=False)
    del ckpt, sd
    return model, cfg


def free_mem(model=None):
    if model is not None:
        model.cpu()
        del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
from model_fock_attention import FockAttentionPARFLM, FockAttentionConfig

# Registry: name -> (ckpt_key, ConfigClass, ModelClass, is_splm)
MODEL_REGISTRY = {
    'Multi-Xi SPLM':  ('splm',      SPLMSARFMassLNMultiXiConfig, ScalarPotentialLMSARFMassLNMultiXi, True),
    'Fock v2.1':      ('fock_v21',  FockMultiXiPARFConfig,       FockMultiXiPARFLM,                  False),
    'Fock Attention': ('fock_attn', FockAttentionConfig,         FockAttentionPARFLM,                False),
}
MODEL_NAMES = list(MODEL_REGISTRY.keys())


def load_model(name):
    ckpt_key, ConfigClass, ModelClass, is_splm = MODEL_REGISTRY[name]
    model, cfg = _load_from_ckpt(ckpt_paths[ckpt_key], ConfigClass, ModelClass, LOGFREQ_PATH)
    model.to(DEVICE).eval()
    model._cfg_cached = cfg
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded {name}: {n:.2f}M params (~{n * 4:.0f} MB VRAM)')
    return model, is_splm


print(f'\nRegistry ready: {MODEL_NAMES}')
print('Models load ONE AT A TIME per stage to minimise peak RAM.')

In [ ]:
# ── Cell 3: Config + paired clean/shuffled dataset ────────────────
# ┌─────────────────────────────────────────────────────────────────┐
# │  EXPERIMENT CONFIG — edit here.                                  │
# └─────────────────────────────────────────────────────────────────┘
PROMPT_LEN   = 96      # P: clean context tokens (identical for clean & corrupted)
CONT_LEN     = 32      # C: continuation tokens (signal is averaged over these)
N_PER_CLASS  = 200     # examples per class (clean / corrupted); paired by story
SEED         = 1234

# Corruption battery. Each name maps to a function (prompt_ids, cont_ids, rng)->new_cont_ids.
# Default: shuffle only. Others kept available for easy extension.
ACTIVE_CORRUPTIONS = ['shuffle']

PROC_BS      = 16      # sequences processed per forward pass (raise on A100/H100)
N_POWER_ITER = 10      # power-iteration steps for K_max (batched HVP)
GAMMA_EFF_SEQS = 64    # clean sequences used to estimate gamma_eff per model

rng_global = np.random.default_rng(SEED)
SEQ_LEN = PROMPT_LEN + CONT_LEN

# ── Corruption functions ──
def corrupt_shuffle(prompt_ids, cont_ids, rng):
    """Permute continuation tokens — identical multiset, broken sequential order."""
    perm = rng.permutation(len(cont_ids))
    return cont_ids[perm]

def corrupt_random_replace(prompt_ids, cont_ids, rng, frac=0.30, vocab=None):
    out = cont_ids.copy()
    k = max(1, int(frac * len(out)))
    pos = rng.choice(len(out), k, replace=False)
    out[pos] = rng.integers(0, vocab, size=k)
    return out

CORRUPTION_FNS = {'shuffle': corrupt_shuffle, 'random_replace': corrupt_random_replace}

# ── Load validation tokens (OOM-safe lightweight path) ──
import pyarrow.parquet as pq
from data_module import (_download_hf_parquet, _resolve_tinystories_shard,
                         _gpt2_tokenize)

DATA_DIR = os.path.join(ARCH_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
val_cache = os.path.join(DATA_DIR, 'tinystories_val_only.npy')
if os.path.exists(val_cache):
    val_ids = np.load(val_cache)
    print(f'Loaded cached val tokens: {len(val_ids):,}')
else:
    val_fname = _resolve_tinystories_shard('data/validation-00000-of-00001')
    vp = _download_hf_parquet('roneneldan/TinyStories', val_fname, 'tinystories_val.parquet')
    val_texts = pq.read_table(vp, columns=['text'])['text'].to_pylist()
    val_ids = _gpt2_tokenize('\n\n'.join(val_texts[:2000]))
    del val_texts
    np.save(val_cache, val_ids)
    print(f'Cached {len(val_ids):,} val tokens')
gc.collect()

VOCAB_SIZE = int(val_ids.max()) + 1  # for random_replace fallback
need = N_PER_CLASS * (SEQ_LEN + 1)
assert len(val_ids) >= need, f'Not enough val tokens ({len(val_ids)}) for {N_PER_CLASS} windows'

# ── Sample N non-overlapping windows, build paired clean/corrupted examples ──
max_start = len(val_ids) - SEQ_LEN - 1
starts = rng_global.choice(max_start, size=N_PER_CLASS, replace=False)

clean_seqs, corrupt_seqs, corrupt_types = [], [], []
for s in starts:
    window = val_ids[s:s + SEQ_LEN].astype(np.int64)
    prompt, cont = window[:PROMPT_LEN], window[PROMPT_LEN:]
    clean_seqs.append(window.copy())
    ctype = ACTIVE_CORRUPTIONS[rng_global.integers(len(ACTIVE_CORRUPTIONS))]
    fn = CORRUPTION_FNS[ctype]
    if ctype == 'random_replace':
        new_cont = fn(prompt, cont, rng_global, vocab=VOCAB_SIZE)
    else:
        new_cont = fn(prompt, cont, rng_global)
    corrupt_seqs.append(np.concatenate([prompt, new_cont]))
    corrupt_types.append(ctype)

# Stack: examples 0..N-1 clean (y=0), N..2N-1 corrupted (y=1)
X_seqs = np.stack(clean_seqs + corrupt_seqs)            # (2N, SEQ_LEN)
y_label = np.concatenate([np.zeros(N_PER_CLASS), np.ones(N_PER_CLASS)]).astype(int)
ex_ctype = ['clean'] * N_PER_CLASS + corrupt_types
CONT_SLICE = slice(PROMPT_LEN, SEQ_LEN)

print(f'Built {len(X_seqs)} examples: {N_PER_CLASS} clean + {N_PER_CLASS} corrupted')
print(f'  corruptions: {ACTIVE_CORRUPTIONS}')
print(f'  P={PROMPT_LEN}  C={CONT_LEN}  seq_len={SEQ_LEN}  vocab~{VOCAB_SIZE}')
del val_ids; gc.collect()

In [ ]:
# ── Cell 4: Shared helpers + per-model gamma_eff measurement ──────

def get_mass(model, x, is_splm):
    """Per-token mass. Returns CPU tensor (B,T) or python float (scalar mass)."""
    if is_splm:
        emb = model._embed(x)
        m = model.compute_mass(x, emb)
    else:
        m = model.compute_mass(x)
    if isinstance(m, torch.Tensor):
        return m.detach().cpu()
    return m


def extract_trajectory(model, x, is_splm):
    """L+1 hidden-state tensors on CPU, plus logits on CPU.
    enable_grad is required: integrate() internally calls autograd for f=-grad V."""
    with torch.enable_grad():
        if is_splm:
            out = model(x, targets=None, return_trajectory=True, return_xi_trajectory=False)
        else:
            out = model(x, targets=None, return_trajectory=True)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    logits_cpu = logits.detach().cpu()
    del traj, out, logits, _loss
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return traj_cpu, logits_cpu


def mass_to_BT(m, like_BT):
    """Broadcast mass to (B,T) on CPU matching like_BT shape."""
    if isinstance(m, torch.Tensor):
        mf = m.squeeze(-1) if m.dim() > 2 else m
        if mf.dim() == 2:
            return mf
        return mf.expand(like_BT.shape)
    return torch.full_like(like_BT, float(m))


def iter_proc_batches(seqs_np, bs):
    """Yield (start, tensor(B,seq_len) on DEVICE) chunks."""
    for i in range(0, len(seqs_np), bs):
        chunk = torch.tensor(seqs_np[i:i + bs], dtype=torch.long, device=DEVICE)
        yield i, chunk


def measure_gamma_eff(model, is_splm, clean_seqs_np, n_seqs):
    """Effective damping from the held-out T-ratio profile:
       gamma_eff = -log( median_ℓ [ mean_tok T_{ℓ+1} / mean_tok T_ℓ ] )."""
    sub = clean_seqs_np[:n_seqs]
    T_sum, T_cnt = None, 0
    for _, x in iter_proc_batches(sub, PROC_BS):
        traj, _ = extract_trajectory(model, x, is_splm)
        L = len(traj) - 1
        m = get_mass(model, x, is_splm)
        if T_sum is None:
            T_sum = np.zeros(L)  # T_1 .. T_L
        for ell in range(1, L + 1):
            v = traj[ell] - traj[ell - 1]
            vn2 = (v ** 2).sum(dim=-1)            # (B,T) CPU
            mf = mass_to_BT(m, vn2)
            T_ell = (0.5 * mf * vn2)
            T_sum[ell - 1] += T_ell.sum().item()
        T_cnt += traj[0].shape[0] * traj[0].shape[1]
        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    T_mean = T_sum / max(T_cnt, 1)               # mean T per layer
    ratios = [T_mean[i + 1] / T_mean[i] for i in range(len(T_mean) - 1)
              if T_mean[i] > 0]
    ratios = [r for r in ratios if r > 0]
    median_ratio = float(np.median(ratios)) if ratios else 1.0
    gamma_eff = -math.log(max(median_ratio, 1e-8))
    return gamma_eff, T_mean.tolist(), ratios


print('═' * 64)
print('GAMMA_EFF MEASUREMENT (held-out T-ratio profile)')
print('═' * 64)
print(f'{"Model":<18}{"gamma_param":>13}{"gamma_eff":>12}{"median T-ratio":>16}')
print('-' * 64)

gamma_info = {}
clean_only = X_seqs[y_label == 0]
for name in MODEL_NAMES:
    model, is_splm = load_model(name)
    g_param = model.gamma.item() if hasattr(model, 'gamma') and hasattr(model.gamma, 'item') \
        else float(getattr(model, 'gamma', 0.0))
    g_eff, T_mean, ratios = measure_gamma_eff(model, is_splm, clean_only, GAMMA_EFF_SEQS)
    med = math.exp(-g_eff)
    gamma_info[name] = {'gamma_param': g_param, 'gamma_eff': g_eff,
                        'T_mean': T_mean, 'median_ratio': med}
    print(f'{name:<18}{g_param:>13.4f}{g_eff:>12.4f}{med:>16.4f}')
    free_mem(model); del model

print('═' * 64)
print('Using gamma_eff for ΔE_expected = -gamma_eff * ||v||^2 * dt')

In [ ]:
# ── Cell 5: Core signal extraction (ΔE_anomaly, K_max, H_softmax) ─
# Per example we average each signal over the CONT_LEN continuation tokens.
# One model at a time; processed in PROC_BS chunks.

def lambda_max_hvp_batched(model, h_bc, xi_bc, n_iter=10):
    """λ_max(∇²V) per position via batched power iteration.
    V_θ is independent per position, so the batched Hessian is block-diagonal
    and per-position HVPs are exact. h_bc:(B,C,d), xi_bc:(B,C,K,d) detached."""
    B, C, d = h_bc.shape
    h_base = h_bc.detach()
    u = torch.randn(B, C, d, device=h_bc.device, dtype=h_bc.dtype)
    u = u / u.norm(dim=-1, keepdim=True).clamp(min=1e-10)
    lam = torch.zeros(B, C, device=h_bc.device, dtype=h_bc.dtype)
    for _ in range(n_iter):
        h_in = h_base.clone().requires_grad_(True)
        V = model.V_theta(xi_bc, h_in)
        gV = torch.autograd.grad(V.sum(), h_in, create_graph=True)[0]
        Hv = torch.autograd.grad((gV * u).sum(), h_in, retain_graph=False)[0].detach()
        lam = (Hv * u).sum(dim=-1)
        u = (Hv / Hv.norm(dim=-1, keepdim=True).clamp(min=1e-10)).detach()
        del h_in, V, gV, Hv
    return lam.abs()


def compute_signals_for_model(model, is_splm, seqs_np, gamma_eff):
    dt = getattr(getattr(model, '_cfg_cached', None), 'dt', 1.0)
    anom_feat, kmax_feat, ent_feat = [], [], []

    for _, x in iter_proc_batches(seqs_np, PROC_BS):
        traj, logits = extract_trajectory(model, x, is_splm)   # CPU
        L = len(traj) - 1
        mid = max(L // 2, 1)
        m = get_mass(model, x, is_splm)

        # reference (B,T) shape for mass broadcast
        ref_BT = (traj[1] - traj[0])[..., 0]
        mf = mass_to_BT(m, ref_BT)                              # (B,T) CPU

        # potential per layer (no grad)
        V_layers = []
        for ell in range(L + 1):
            h = traj[ell].to(DEVICE)
            with torch.no_grad():
                xis = model.xi_module(h.detach())
                V = model.V_theta(xis, h).squeeze(-1).cpu()     # (B,T)
            V_layers.append(V)
            del h, xis
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

        # kinetic + |v|^2 per layer (ell=1..L), and energy H
        vn2_layers = [None]
        H = [V_layers[0]]                                       # H_0 = V_0 (T_0=0)
        for ell in range(1, L + 1):
            v = traj[ell] - traj[ell - 1]
            vn2 = (v ** 2).sum(dim=-1)                          # (B,T) CPU
            vn2_layers.append(vn2)
            H.append(0.5 * mf * vn2 + V_layers[ell])

        # ΔE_anomaly per token, averaged over layers
        anom = torch.zeros_like(H[0])
        for ell in range(1, L + 1):
            dE_obs = H[ell] - H[ell - 1]
            dE_exp = -gamma_eff * vn2_layers[ell] * dt
            anom += (dE_obs - dE_exp).abs()
        anom = anom / L                                         # (B,T)

        # H_softmax (next-token entropy)
        probs = torch.softmax(logits.float(), dim=-1)
        ent = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)   # (B,T) CPU
        del probs, logits

        # K_max at mid layer on continuation tokens (batched HVP)
        h_mid = traj[mid][:, CONT_SLICE, :].to(DEVICE)          # (B,C,d)
        with torch.no_grad():
            xi_mid = model.xi_module(h_mid.detach()).detach()
        lam = lambda_max_hvp_batched(model, h_mid, xi_mid, N_POWER_ITER).cpu()
        v_mid2 = vn2_layers[mid][:, CONT_SLICE]                 # (B,C)
        T_mid = (0.5 * mf[:, CONT_SLICE] * v_mid2).abs().clamp(min=1e-8)
        K = lam / (2.0 * T_mid)                                 # (B,C)
        del h_mid, xi_mid, lam

        # per-example aggregation over continuation tokens
        anom_feat.extend(anom[:, CONT_SLICE].mean(dim=1).tolist())
        ent_feat.extend(ent[:, CONT_SLICE].mean(dim=1).tolist())
        kmax_feat.extend(K.mean(dim=1).tolist())

        del traj, V_layers, vn2_layers, H, anom, ent, K
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    return {'anomaly': np.array(anom_feat),
            'kmax':    np.array(kmax_feat),
            'entropy': np.array(ent_feat)}


print('═' * 64)
print('SIGNAL EXTRACTION')
print('═' * 64)

signals = {}
for name in MODEL_NAMES:
    t0 = time.time()
    print(f'\n── {name} ──')
    model, is_splm = load_model(name)
    g_eff = gamma_info[name]['gamma_eff']
    sig = compute_signals_for_model(model, is_splm, X_seqs, g_eff)
    signals[name] = sig
    # quick clean-vs-corrupt mean check
    for s in ['anomaly', 'kmax', 'entropy']:
        c0 = sig[s][y_label == 0].mean()
        c1 = sig[s][y_label == 1].mean()
        print(f'  {s:<8}  clean={c0:.4e}  corrupt={c1:.4e}  Δ={c1 - c0:+.4e}')
    print(f'  ({time.time() - t0:.1f}s)')
    free_mem(model); del model

print('\n✓ Signal extraction complete.')

In [ ]:
# ── Cell 6: Detector evaluation (AUROC + probes + ECE) ────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve

FEATURE_SETS = {
    'ΔE_anomaly':        ['anomaly'],
    'K_max':             ['kmax'],
    'H_softmax (base)':  ['entropy'],
    '[ΔE, K_max]':       ['anomaly', 'kmax'],
    '[ΔE, K_max, H]':    ['anomaly', 'kmax', 'entropy'],
}


def ece_score(y_true, p_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p_pred > lo) & (p_pred <= hi) if i > 0 else (p_pred >= lo) & (p_pred <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p_pred[m].mean()
        ece += (m.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)


def raw_auroc(signal, y):
    """Direction-agnostic single-signal AUROC (orient so AUROC >= 0.5)."""
    a = roc_auc_score(y, signal)
    return a if a >= 0.5 else 1.0 - a


detector_results = {}
oof_curves = {}   # name -> {feat_label: (fpr, tpr, auc)}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('═' * 78)
print('DETECTOR AUROC  (5-fold CV logistic probe; raw = single-signal direction-agnostic)')
print('═' * 78)

for name in MODEL_NAMES:
    sig = signals[name]
    feat_mat = {k: sig[k].reshape(-1, 1) for k in ['anomaly', 'kmax', 'entropy']}
    res, curves = {}, {}
    print(f'\n── {name}  (gamma_eff={gamma_info[name]["gamma_eff"]:.4f}) ──')
    print(f'  {"signal":<20}{"raw AUROC":>11}{"probe AUROC":>13}{"ECE":>8}')
    print('  ' + '-' * 50)
    for label, cols in FEATURE_SETS.items():
        X = np.hstack([feat_mat[c] for c in cols])
        pipe = make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=1000, class_weight='balanced'))
        oof = cross_val_predict(pipe, X, y_label, cv=cv, method='predict_proba')[:, 1]
        probe_auc = roc_auc_score(y_label, oof)
        ece = ece_score(y_label, oof)
        raw = raw_auroc(sig[cols[0]], y_label) if len(cols) == 1 else float('nan')
        fpr, tpr, _ = roc_curve(y_label, oof)
        curves[label] = (fpr.tolist(), tpr.tolist(), probe_auc)
        res[label] = {'raw_auroc': raw, 'probe_auroc': probe_auc, 'ece': ece}
        raw_str = f'{raw:>11.4f}' if not math.isnan(raw) else f'{"—":>11}'
        print(f'  {label:<20}{raw_str}{probe_auc:>13.4f}{ece:>8.4f}')
    detector_results[name] = res
    oof_curves[name] = curves

    # ── core hypothesis verdict ──
    a_anom = res['ΔE_anomaly']['probe_auroc']
    a_base = res['H_softmax (base)']['probe_auroc']
    a_joint = res['[ΔE, K_max]']['probe_auroc']
    verdict = 'PASS' if a_anom > a_base else 'fail'
    print(f'  → ΔE_anomaly ({a_anom:.3f}) vs H_softmax ({a_base:.3f}): {verdict}'
          f'   |  [ΔE,K_max] joint = {a_joint:.3f}')

print('\n' + '═' * 78)

In [ ]:
# ── Cell 7: Plots — ROC curves, distributions, AUROC bars ─────────
PALETTE = {'Multi-Xi SPLM': '#3B82F6', 'Fock v2.1': '#22C55E', 'Fock Attention': '#F59E0B'}
FEAT_COLORS = {
    'ΔE_anomaly': '#3B82F6', 'K_max': '#8B5CF6', 'H_softmax (base)': '#9CA3AF',
    '[ΔE, K_max]': '#EF4444', '[ΔE, K_max, H]': '#111827',
}

# ── Fig 1: ROC curves (one panel per model) ──
fig1, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 5))
if len(MODEL_NAMES) == 1:
    axes = [axes]
for ax, name in zip(axes, MODEL_NAMES):
    for label, (fpr, tpr, auc) in oof_curves[name].items():
        ax.plot(fpr, tpr, lw=2, color=FEAT_COLORS.get(label, None),
                label=f'{label} ({auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', lw=1, alpha=0.5)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
    ax.legend(fontsize=8, loc='lower right')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Experiment B — ROC: inconsistency detection (shuffle task)',
             fontweight='bold', y=1.03)
plt.tight_layout()
fig1.savefig(DRIVE_RESULTS / 'expB_roc_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ── Fig 2: signal distributions (rows: ΔE_anomaly, H_softmax) × models ──
fig2, axes2 = plt.subplots(2, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 8))
if len(MODEL_NAMES) == 1:
    axes2 = axes2.reshape(2, 1)
for j, name in enumerate(MODEL_NAMES):
    sig = signals[name]
    for r, skey, sname in [(0, 'anomaly', 'ΔE_anomaly'), (1, 'entropy', 'H_softmax')]:
        ax = axes2[r, j]
        v0 = sig[skey][y_label == 0]; v1 = sig[skey][y_label == 1]
        lo = min(v0.min(), v1.min()); hi = max(v0.max(), v1.max())
        bins = np.linspace(lo, hi, 40)
        ax.hist(v0, bins=bins, alpha=0.6, color='#22C55E', label='clean', density=True)
        ax.hist(v1, bins=bins, alpha=0.6, color='#EF4444', label='corrupt', density=True)
        ax.set_title(f'{name}\n{sname}', fontsize=9, fontweight='bold')
        ax.set_xlabel(sname); ax.set_ylabel('density'); ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Signal distributions: clean vs corrupted', fontweight='bold', y=1.02)
plt.tight_layout()
fig2.savefig(DRIVE_RESULTS / 'expB_signal_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ── Fig 3: grouped AUROC bar chart (probe AUROC) ──
fig3, ax3 = plt.subplots(figsize=(11, 5))
labels = list(FEATURE_SETS.keys())
xb = np.arange(len(MODEL_NAMES))
w = 0.16
for k, label in enumerate(labels):
    vals = [detector_results[n][label]['probe_auroc'] for n in MODEL_NAMES]
    ax3.bar(xb + (k - len(labels) / 2) * w + w / 2, vals, w,
            label=label, color=FEAT_COLORS.get(label, None))
ax3.axhline(0.5, color='gray', ls='--', alpha=0.6, label='chance')
ax3.set_xticks(xb); ax3.set_xticklabels(MODEL_NAMES)
ax3.set_ylabel('Probe AUROC'); ax3.set_ylim(0.4, 1.0)
ax3.set_title('Detector AUROC by signal and model', fontweight='bold')
ax3.legend(fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
plt.tight_layout()
fig3.savefig(DRIVE_RESULTS / 'expB_auroc_bars.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print('Saved figures:')
for f in ['expB_roc_curves.png', 'expB_signal_distributions.png', 'expB_auroc_bars.png']:
    print(f'  {DRIVE_RESULTS / f}')

In [ ]:
# ── Cell 8: Persist JSON report + per-example signals ─────────────
report = {
    'experiment': 'dissipation_anomaly_inconsistency_detector',
    'task': 'controlled_story_consistency_shuffle',
    'models': MODEL_NAMES,
    'config': {
        'prompt_len': PROMPT_LEN, 'cont_len': CONT_LEN,
        'n_per_class': N_PER_CLASS, 'active_corruptions': ACTIVE_CORRUPTIONS,
        'n_power_iter': N_POWER_ITER, 'proc_bs': PROC_BS,
        'seed': SEED, 'device': DEVICE,
        'gamma_source': 'gamma_eff (held-out T-ratio median)',
    },
    'gamma_info': gamma_info,
    'detector_results': detector_results,
    'roc_curves': oof_curves,
}

report_path = DRIVE_RESULTS / 'hallucination_detector_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f'Report saved: {report_path}')

# Per-example raw signals (for downstream re-analysis / soft-decoding work)
per_example = {
    'y_label': y_label.tolist(),
    'corruption_type': ex_ctype,
    'signals': {name: {k: signals[name][k].tolist() for k in ['anomaly', 'kmax', 'entropy']}
                for name in MODEL_NAMES},
}
sig_path = DRIVE_RESULTS / 'per_example_signals.json'
with open(sig_path, 'w') as f:
    json.dump(per_example, f, default=str)
print(f'Per-example signals: {sig_path}')

print(f'\nAll files in {DRIVE_RESULTS}:')
for p in sorted(DRIVE_RESULTS.iterdir()):
    size = p.stat().st_size
    unit = 'KB' if size > 1024 else 'B'
    val = size / 1024 if size > 1024 else size
    print(f'  {p.name:42s}  {val:8.1f} {unit}')

print('\n' + '═' * 64)
print('EXPERIMENT B — DISSIPATION-ANOMALY DETECTOR — COMPLETE')
print('═' * 64)